## Load NSE Data into PostgreSQL
This notebook reads all 19 annual NSE CSV files, cleans them and writes them to the database.
Prerequisites:
pip install -r requirements.txt
docker-compose up -d
docker exec -i ...
all csv files in raw

In [ ]:
import os, sys, logging
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

DB_URL = (
    f"postgresql://{os.getenv('DB_USER','postgres')}:"
    f"{os.getenv('DB_PASSWORD','nse_password')}@"
    f"{os.getenv('DB_HOST','localhost')}:"
    f"{os.getenv('DB_PORT','5432')}/"
    f"{os.getenv('DB_NAME','nse_portfolio')}"
)
RAW = Path("data/raw")
print("Imports OK")

False

## Define the stock environment
67 qualifying equities sorted alphabetically

In [ ]:
ALL_TICKERS = sorted([
    "ABSA","ARM","BAMB","BAT","BKG","BOC","BRIT","CABL","CARB","CGEN",
    "CIC","COOP","CRWN","CTUM","DCON","DTK","EABL","EGAD","EQTY","EVRD",
    "FAHR","FTGH","GLD","HAFR","HBE","HFCK","IMH","JUB","KAPC","KCB",
    "KEGN","KNRE","KPLC","KPLC-P4","KPLC-P7","KQ","KUKZ","KURV","LAPR",
    "LBTY","LIMT","LKL","MSC","NBK","NBV","NCBA","NMG","NSE","OCH",
    "ORCH","PORT","SASN","SBIC","SCAN","SCBK","SCOM","SGL","SLAM","SMER",
    "TCL","TOTL","TPSE","UCHM","UMME","UNGA","WTK","XPRS",
])
print(f"Stock universe: {len(ALL_TICKERS)} qualifying equities")

| Problem | Years | Fix |
# |---|---|---|
# | Date `"1/2/2007"` (M/D/Y) | 2007–2012 | `format='%m/%d/%Y'` |
# | Date `"2-Jan-24"` (D-Mon-Y) | 2013–2025 | `format='%d-%b-%y'` |
# | `"Adjust"` vs `"Adjusted"` vs `"Adjusted Price"` | All | Map all to `adj_close` |
# | Volume `"416,380,000"` (commas) | All | Strip commas before `to_numeric` |
# | Missing adj_close shown as `"-"` | All | Fall back to raw close |

## Helper functions to normalise columns, parse dates

In [ ]:
def normalise_cols(df):
    rn = {}
    for c in df.columns:
        cu = c.upper().strip()
        if   cu == "DATE":                              rn[c] = "date"
        elif cu == "CODE":                              rn[c] = "ticker"
        elif cu == "NAME":                              rn[c] = "name"
        elif cu == "12M LOW":                           rn[c] = "low_52w"
        elif cu == "12M HIGH":                          rn[c] = "high_52w"
        elif cu == "DAY LOW":                           rn[c] = "low"
        elif cu == "DAY HIGH":                          rn[c] = "high"
        elif cu == "DAY PRICE":                         rn[c] = "close"
        elif cu == "PREVIOUS":                          rn[c] = "prev_close"
        elif cu == "CHANGE":                            rn[c] = "change_val"
        elif cu in ("CHANGE%", "CHANGE %"):             rn[c] = "change_pct"
        elif cu == "VOLUME":                            rn[c] = "volume"
        elif cu in ("ADJUST", "ADJUSTED", "ADJUSTED PRICE"): rn[c] = "adj_close"
    return df.rename(columns=rn)

def parse_date(series, year):
    if year <= 2012:
        return pd.to_datetime(series, format="%m/%d/%Y", errors="coerce")
    parsed = pd.to_datetime(series, format="%d-%b-%y", errors="coerce")
    if parsed.isna().mean() > 0.05:
        parsed = pd.to_datetime(series, dayfirst=True, errors="coerce")
    return parsed

def to_num(series):
    return (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace("-", np.nan)
        .replace("", np.nan)
        .pipe(pd.to_numeric, errors="coerce")
    )

print("Helper functions defined")

## Load all 19 CSV files

In [ ]:
csv_files = sorted(RAW.glob("NSE_data_all_stocks_*.csv"))
print(f"Found {len(csv_files)} CSV files")

frames = []
for path in csv_files:
    year = int(path.stem.split("_")[-1])
    df   = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    df.columns = df.columns.str.strip()
    df   = normalise_cols(df)
    df["date"]   = parse_date(df["date"], year)
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    frames.append(df)
    print(f"  {year}: {len(df):,} rows, {df['ticker'].nunique()} tickers")

raw = pd.concat(frames, ignore_index=True)
print(f"\nCombined: {len(raw):,} rows")

## Filtering and Cleaning

In [ ]:
df = raw[raw["ticker"].isin(ALL_TICKERS)].copy()
df = df.dropna(subset=["date"])

for col in ["close","high","low","prev_close","volume","adj_close","low_52w","high_52w"]:
    if col in df.columns:
        df[col] = to_num(df[col])

# Fall back to raw close when adj_close is missing
missing = df["adj_close"].isna().sum()
df["adj_close"] = df["adj_close"].fillna(df["close"])
print(f"Filled {missing:,} missing adj_close values with raw close")

df = df[df["close"] > 0].copy()
df = df.drop_duplicates(subset=["date", "ticker"], keep="last")
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
print(f"After cleaning: {len(df):,} rows, {df['ticker'].nunique()} tickers")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

Forward filling for short gaps is chosen for this. This is a range of about 1-5 days. Gaps longer than 5 days remain unfilled and the environment treats those as days when the stock is unavailable

In [ ]:
calendar = pd.date_range(df["date"].min(), df["date"].max(), freq="B")
num_cols  = ["close","high","low","adj_close","prev_close","volume","low_52w","high_52w"]
filled_frames = []

for ticker in ALL_TICKERS:
    sub = df[df["ticker"] == ticker].set_index("date").sort_index()
    if sub.empty:
        continue
    sub = sub.reindex(calendar)
    sub["ticker"] = ticker
    cols = [c for c in num_cols if c in sub.columns]
    sub[cols] = sub[cols].ffill(limit=5)
    sub = sub.dropna(subset=["close"])
    filled_frames.append(sub.reset_index().rename(columns={"index": "date"}))

filled = pd.concat(filled_frames, ignore_index=True)
print(f"After gap-fill: {len(filled):,} rows, {filled['ticker'].nunique()} tickers")

## Compute observation Features
For each stock on each trading day, the model uses seven features that are calculated exclusively from historical information, ensuring there is no lookahead bias. These features include **1-day log return** ((\log(p_t/p_{t-1}))), which captures short-term noise and reversal effects; **5-day log return** ((\log(p_t/p_{t-5}))), which measures weekly momentum; **20-day log return** ((\log(p_t/p_{t-20}))), representing monthly momentum; and **60-day log return** ((\log(p_t/p_{t-60}))), which captures the longer-term quarterly trend. The model also incorporates **20-day volatility**, calculated as the standard deviation of daily returns over the previous 20 trading days, to provide a measure of risk since higher volatility generally reduces risk-adjusted performance as measured by the Sharpe ratio. To account for market frictions, two liquidity indicators are included: the **Corwin–Schultz bid-ask spread estimator** (Corwin & Schultz, 2012), which proxies transaction costs and liquidity constraints, and the **Amihud illiquidity measure** (Amihud, 2002), which captures price impact and trading difficulty. Among these features, the **60-day return is particularly important** because it helps the agent determine whether a trend is strong enough to overcome the approximately 4.16% round-trip transaction cost in the Nairobi Securities Exchange (NSE). For example, a stock generating an average return of 0.07% per day over 60 trading days produces a cumulative return of roughly 4.2%, just exceeding the transaction-cost breakeven point. Consequently, the 60-day return provides a crucial signal that helps the agent evaluate whether entering and holding a position is likely to generate sufficient profit after costs.


In [ ]:
feature_frames = []

for i, ticker in enumerate(ALL_TICKERS):
    sub = filled[filled["ticker"] == ticker].sort_values("date").copy()
    if len(sub) < 65:
        continue
    p = sub["adj_close"]

    sub["return_1d"]  = np.log(p / p.shift(1))
    sub["return_5d"]  = np.log(p / p.shift(5))
    sub["return_20d"] = np.log(p / p.shift(20))
    sub["return_60d"] = np.log(p / p.shift(60))
    sub["vol_20d"]    = sub["return_1d"].rolling(20, min_periods=10).std()

    # Corwin-Schultz spread
    h  = sub["high"].replace(0, np.nan)
    l  = sub["low"].replace(0, np.nan)
    bt = np.log(h / l) ** 2
    bt1 = bt.shift(-1)
    h2 = h.rolling(2).max()
    l2 = l.rolling(2).min()
    gm = np.log(h2 / l2.replace(0, np.nan)) ** 2
    k  = 3 - 2 * np.sqrt(2)
    ba = (bt + bt1) / 2
    with np.errstate(invalid="ignore", divide="ignore"):
        al = (np.sqrt(2 * ba) - np.sqrt(ba)) / k - np.sqrt(gm / k)
        sp = 2 * (np.exp(al) - 1) / (1 + np.exp(al))
    sub["cs_spread"] = np.clip(np.nan_to_num(sp, nan=0.0), 0.0, 0.30)

    # Amihud illiquidity
    tv = sub["volume"] * p
    sub["amihud"] = (
        sub["return_1d"].abs() / tv.replace(0, np.nan)
    ).rolling(20, min_periods=10).mean()

    feature_frames.append(sub[[
        "date", "ticker",
        "return_1d", "return_5d", "return_20d", "return_60d",
        "vol_20d", "cs_spread", "amihud",
    ]])

    if (i + 1) % 15 == 0:
        print(f"  Features: {i+1}/{len(ALL_TICKERS)} stocks done")

features = pd.concat(feature_frames, ignore_index=True)
features  = features.replace([np.inf, -np.inf], np.nan)
print(f"Features computed: {len(features):,} rows")

## Connect to DB

In [ ]:
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Database connection OK")
except Exception as e:
    print(f"ERROR: {e}")
    print("Ensure Docker is running:  docker-compose up -d")
    raise

## Write Prices to DB

In [ ]:
def batch_write(df, engine, sql, batch_size=5000):
    rows = df.where(pd.notnull(df), None).to_dict(orient="records")
    total = len(rows)
    for i in range(0, total, batch_size):
        with engine.begin() as conn:
            conn.execute(text(sql), rows[i:i + batch_size])
        print(f"  {min(i + batch_size, total):,} / {total:,}", end="\r")
    print()

price_cols = ["date","ticker","name","high","low","close","adj_close",
              "volume","high_52w","low_52w","prev_close","change_val","change_pct"]
price_df   = filled[[c for c in price_cols if c in filled.columns]].copy()

print("Writing prices ...")
batch_write(price_df, engine, """
    INSERT INTO nse_prices
        (date,ticker,name,high,low,close,adj_close,volume,
         high_52w,low_52w,prev_close,change_val,change_pct)
    VALUES
        (:date,:ticker,:name,:high,:low,:close,:adj_close,:volume,
         :high_52w,:low_52w,:prev_close,:change_val,:change_pct)
    ON CONFLICT (date,ticker) DO UPDATE SET
        close=EXCLUDED.close, adj_close=EXCLUDED.adj_close,
        high=EXCLUDED.high, low=EXCLUDED.low, volume=EXCLUDED.volume
""")
print("Prices written.")

## Write Features to DB

In [ ]:
print("Writing features ...")
batch_write(features, engine, """
    INSERT INTO nse_features
        (date,ticker,return_1d,return_5d,return_20d,return_60d,
         vol_20d,cs_spread,amihud)
    VALUES
        (:date,:ticker,:return_1d,:return_5d,:return_20d,:return_60d,
         :vol_20d,:cs_spread,:amihud)
    ON CONFLICT (date,ticker) DO UPDATE SET
        return_60d=EXCLUDED.return_60d,
        cs_spread=EXCLUDED.cs_spread,
        amihud=EXCLUDED.amihud
""")
print("Features written.")

In [ ]:
##validation report

In [ ]:
with engine.connect() as conn:
    rows = conn.execute(text(
        "SELECT ticker, COUNT(*) r, MIN(date) f, MAX(date) l "
        "FROM nse_prices GROUP BY ticker ORDER BY r DESC"
    )).fetchall()

print(f"{'Ticker':<12} {'Rows':>6}  {'First':>12}  {'Last':>12}")
print("-" * 48)
for r in rows:
    print(f"{r[0]:<12} {r[1]:>6,}  {str(r[2]):>12}  {str(r[3]):>12}")

with engine.connect() as conn:
    fc = conn.execute(text("SELECT COUNT(*) FROM nse_features")).scalar()

print(f"\nFeature rows: {fc:,}")
print(f"Tickers loaded: {len(rows)} / {len(ALL_TICKERS)}")
print("\nDatabase ready. Open training/02_train.ipynb next.")